LOAD DATASET

In [26]:
import pandas as pd

df = pd.read_csv("../data/raw_dataset.csv", encoding="latin1")

print("Shape:", df.shape)
print(df.head())
print(df["label"].value_counts())
print("\nPercentage distribution:\n")
print(df["label"].value_counts(normalize=True))
df["label"] = df["label"].str.strip()

Shape: (244, 3)
                       title  \
0             Reverse String   
1           Valid Palindrome   
2        Valid Palindrome II   
3    Valid Word Abbreviation   
4  Merge Strings Alternately   

                                         description         label  
0  You are given an array of characters which rep...  Two Pointers  
1  Given a string s, return true if it is a palin...  Two Pointers  
2  You are given a string s, return true if the s...  Two Pointers  
3  A string can be shortened by replacing any num...  Two Pointers  
4  You are given two strings, word1 and word2. Co...  Two Pointers  
label
Two Pointers           37
Sliding Window         37
Dynamic Programming    36
Greedy                 35
Binary Search          33
Graphs                 33
Heap/Priority Queue    27
Name: count, dtype: int64

Percentage distribution:

label
Two Pointers           0.155462
Sliding Window         0.155462
Dynamic Programming    0.151261
Greedy                 0.147059
Bi

CLEAN TEXT

In [35]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.encode("ascii", "ignore").decode()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)

In [36]:
print("Nulls in description:", df["clean_description"].isnull().sum())
print("Nulls in label:", df["label"].isnull().sum())
print(df[df["clean_description"].isnull()])


Nulls in description: 0
Nulls in label: 6
Empty DataFrame
Columns: [title, description, label, clean_description]
Index: []


In [37]:
df = df.dropna(subset=["clean_description", "label"])
print(df.isnull().sum())

title                0
description          0
label                0
clean_description    0
dtype: int64


TRAIN/TEST SPLIT

In [38]:
from sklearn.model_selection import train_test_split

X = df["clean_description"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 190
Test size: 48


MAJORITY BASELINE

In [16]:
from sklearn.metrics import accuracy_score

majority_class = y_train.value_counts().idxmax()
y_majority_pred = [majority_class] * len(y_test)

majority_accuracy = accuracy_score(y_test, y_majority_pred)

print("Majority class:", majority_class)
print("Majority baseline accuracy:", round(majority_accuracy, 3))

Majority class: Two Pointers
Majority baseline accuracy: 0.146


TF-IDF

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    stop_words="english",
    max_features=8000,
    min_df=2              # ignore extremely rare words
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (190, 1574)


MODEL COMPARISION

In [ ]:
# LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1500,
    class_weight="balanced",
    C=1.0
)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

In [39]:
# LINEAR SVC
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

model = LinearSVC(class_weight="balanced")

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Linear SVM Accuracy:", round(accuracy, 3))

Linear SVM Accuracy: 0.438


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


FULL REPORT

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

                     precision    recall  f1-score   support

      Binary Search       0.43      0.43      0.43         7
Dynamic Programming       0.25      0.29      0.27         7
             Graphs       1.00      0.57      0.73         7
             Greedy       0.44      0.57      0.50         7
Heap/Priority Queue       0.40      0.40      0.40         5
     Sliding Window       0.56      0.62      0.59         8
       Two Pointers       0.33      0.29      0.31         7

           accuracy                           0.46        48
          macro avg       0.49      0.45      0.46        48
       weighted avg       0.49      0.46      0.46        48

[[3 0 0 1 1 1 1]
 [0 2 0 2 1 0 2]
 [1 2 4 0 0 0 0]
 [0 1 0 4 0 2 0]
 [1 0 0 0 2 1 1]
 [1 0 0 2 0 5 0]
 [1 3 0 0 1 0 2]]


In [23]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Improved Accuracy:", round(accuracy, 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Improved Accuracy: 0.458

Classification Report:

                     precision    recall  f1-score   support

      Binary Search       0.43      0.43      0.43         7
Dynamic Programming       0.25      0.29      0.27         7
             Graphs       1.00      0.57      0.73         7
             Greedy       0.44      0.57      0.50         7
Heap/Priority Queue       0.40      0.40      0.40         5
     Sliding Window       0.56      0.62      0.59         8
       Two Pointers       0.33      0.29      0.31         7

           accuracy                           0.46        48
          macro avg       0.49      0.45      0.46        48
       weighted avg       0.49      0.46      0.46        48



In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

for C_value in [0.1, 0.5, 1.0, 2.0, 5.0]:
    
    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        C=C_value
    )
    
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    
    print(f"C={C_value} → Accuracy={round(acc,3)}")

C=0.1 → Accuracy=0.438
C=0.5 → Accuracy=0.458
C=1.0 → Accuracy=0.458
C=2.0 → Accuracy=0.438
C=5.0 → Accuracy=0.438
